# КТ2. Изучение подходов и работа с данными

## 1. Обзор подходов

### **Neural ODE (Chen et al., 2018)**

Базовый детерминированный подход — непрерывная глубокая сеть как ODE:

$$\frac{dX_t}{dt} = f_\theta(X_t, t), \qquad X_T = X_0 + \int_0^T f_\theta(X_t, t)\, dt$$

![Neural ODE vs ResNet](https://rkevingibson.github.io/img/ode_networks_1.png)

Ключевая идея: вместо фиксированного числа слоёв — непрерывная динамика, а ODE solver используется как вычислительный примитив. Обучение через **метод сопряжённых состояний** (adjoint method) — backprop через любой ODE solver за $O(1)$ памяти.

**Ограничение:** детерминированная модель — одно начальное условие даёт одну траекторию. Не позволяет моделировать стохастические данные или неопределённость.

### **Deep Latent Gaussian Models (Rezende et al., 2014)**

Генеративная модель — variational autoencoder с глубокой многошаговой структурой латентной переменной:

$$X_i = X_{i-1} + b_i(X_{i-1}) + \sigma_i Z_i, \qquad Z_i \overset{\text{i.i.d.}}{\sim} N(0, I)$$

![](https://figures.semanticscholar.org/484ad17c926292fbe0d5211540832a8c8a8e958b/2-Figure1-1.png)


Вариационный вывод через stochastic backpropagation (reparametrization trick). **Neural SDE — это предел этой модели** при $k \to \infty$, $\Delta t \to 0$.

### **Neural SDE — Tzen & Raginsky (2019)**

Основная идея статьи – Neural SDE — это математически строгий предел DLGM (когда число слоёв уходит в бесконечность).

Ключевое свойство DLGM — итеративная структура с дифференцируемыми переходами между слоями. Случайность вынесена в фиксированный шум ZiZ_i
Zi​, всё остальное дифференцируемо. Это позволяет считать градиенты обычным backprop — такой подход называется
stochastic backpropagation. В Neural SDE та же идея переходит в непрерывное время: дифференцируемые шаги становятся интегралом Ито, а stochastic backpropagation — теорией стохастических потоков.

Основная статья проекта. Объединяет идеи Neural ODE и DLGM:

| | Neural ODE | DLGM | Neural SDE |
|---|---|---|---|
| Динамика | детерминированная | дискретная стохастическая | непрерывная стохастическая |
| Латентное пространство | $\mathbb{R}^d$ | $(\mathbb{R}^d)^k$ | пространство Винера |
| Вывод | нет | mean-field VI | Gibbs + Girsanov |
| Градиенты | adjoint | reparametrization | stochastic flows |